In [ ]:
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings, os, math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
np.random.seed(42)
print("✅ Imports OK")

✅ Imports OK


In [ ]:
"""
╔══════════════════════════════════════════════════════════════════════╗
║   SÉRIE TEMPORELLE – IMMOBILIER TUNISIE  (DONNÉES RÉELLES)          ║
║   Dataset : dataset_immobilier_clean.csv  –  42 713 annonces        ║
║   Modèles : ARIMA · SARIMA · Prophet · GRU · RNN · LSTM             ║
║   DSO : Prédire le meilleur moment et lieu pour acheter / vendre     ║
╚══════════════════════════════════════════════════════════════════════╝
"""

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import warnings, os, math
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error

warnings.filterwarnings("ignore")
np.random.seed(42)

OUTPUT_DIR = "/mnt/user-data/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

PALETTE = {
    "primary":   "#1A535C",
    "secondary": "#4ECDC4",
    "accent":    "#FF6B35",
    "warn":      "#FFE66D",
    "dark":      "#1A1A2E",
    "light":     "#F7F7F7",
    "red":       "#E63946",
    "green":     "#2DC653",
    "purple":    "#9B5DE5",
    "orange":    "#F77F00",
    "blue":      "#0077B6",
}
MONTH_NAMES = ["Jan","Fév","Mar","Avr","Mai","Jui","Jul","Aoû","Sep","Oct","Nov","Déc"]

plt.rcParams.update({
    "figure.facecolor": PALETTE["light"],
    "axes.facecolor":   PALETTE["light"],
    "font.family":      "DejaVu Sans",
    "axes.titlesize":   13,
    "axes.labelsize":   11,
    "grid.alpha":       0.3,
})

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 0.  CHARGEMENT ET PRÉPARATION DES VRAIES DONNÉES
# ══════════════════════════════════════════════════════════════════════════════
import pandas as pd
import os

from google.colab import files

# Téléchargez à nouveau le fichier
print("Veuillez télécharger le fichier:")
uploaded = files.upload()

# Obtenir le nom du fichier téléchargé
file_name = list(uploaded.keys())[0]
print(f"Fichier détecté : {file_name}")

# Lire le fichier selon son extension
if file_name.endswith('.csv'):
    df_raw = pd.read_csv(file_name, encoding='utf-8')
    print("✅ Fichier CSV chargé")
elif file_name.endswith('.xlsx'):
    df_raw = pd.read_excel(file_name)
    print("✅ Fichier Excel chargé")
else:
    raise ValueError("Format de fichier non supporté. Utilisez .csv ou .xlsx")

print(f"   Dataset chargé : {df_raw.shape[0]:,} lignes × {df_raw.shape[1]} colonnes")
print(f"   Colonnes disponibles : {df_raw.columns.tolist()}")

# ── Nettoyage ──────────────────────────────────────────────────────────────
# Vérifier les noms de colonnes (adaptez selon votre fichier)
date_col = 'dateAnnonce' if 'dateAnnonce' in df_raw.columns else 'date'
price_col = 'price' if 'price' in df_raw.columns else 'prix'
gov_col = 'gouvernorat' if 'gouvernorat' in df_raw.columns else 'gouvernorat'

print(f"\n   Utilisation des colonnes : date='{date_col}', prix='{price_col}', gouvernorat='{gov_col}'")

df_raw[date_col] = pd.to_datetime(df_raw[date_col], errors='coerce')
df_raw[price_col] = pd.to_numeric(df_raw[price_col], errors='coerce')

# Filtres qualité
df = df_raw.dropna(subset=[date_col, price_col]).copy()
df = df[df[price_col] > 1000]           # retirer prix aberrants < 1000 DT
df = df[df[price_col] < 2_000_000]      # retirer outliers extrêmes
df = df[df[date_col] >= '2022-01-01']   # période avec volume suffisant

print(f"\n   Après filtrage (date + prix + période 2022+) : {len(df):,} annonces")
print(f"   Période : {df[date_col].min().date()} → {df[date_col].max().date()}")
print(f"   Prix médian : {df[price_col].median():,.0f} DT")

# ── Agrégation mensuelle ────────────────────────────────────────────────────
# Série nationale : prix médian mensuel
national_monthly = (df.set_index(date_col)
                      .resample('MS')[price_col]
                      .median()
                      .reset_index())
national_monthly.columns = ['date', 'prix_median']
national_monthly = national_monthly.set_index('date')

# Gouvernorats avec couverture suffisante (≥ 18 mois)
GOUVERNORATS_RETENUS = ['Ariana', 'Ben Arous', 'Bizerte', 'Manouba',
                        'Monastir', 'Nabeul', 'Sfax', 'Sousse', 'Tunis', 'Zaghouan']

gov_monthly = {}
for gov in GOUVERNORATS_RETENUS:
    sub = df[df[gov_col].str.strip().str.title() == gov.title()]
    if len(sub) == 0:
        # Essai sans normalisation de casse
        sub = df[df[gov_col] == gov]
    if len(sub) > 0:
        s = (sub.set_index(date_col)
                .resample('MS')[price_col]
                .median()
                .reindex(national_monthly.index))
        # Interpoler les mois manquants
        s = s.interpolate(method='linear').bfill().ffill()
        gov_monthly[gov] = s
    else:
        print(f"   ⚠️ Aucune donnée pour le gouvernorat : {gov}")

series_main = national_monthly['prix_median'].copy()
n = len(series_main)

print(f"\n   Série temporelle nationale : {n} mois")
print(f"   Gouvernorats retenus avec données : {', '.join(gov_monthly.keys())}")
print(f"\n   Aperçu série nationale (prix médian DT) :")
for d, v in series_main.items():
    print(f"     {d.strftime('%Y-%m')} : {v:>10,.0f} DT")

Veuillez télécharger le fichier:


Saving dataset_immobilier_clean (1).csv to dataset_immobilier_clean (1).csv
Fichier détecté : dataset_immobilier_clean (1).csv
✅ Fichier CSV chargé
   Dataset chargé : 42,713 lignes × 26 colonnes
   Colonnes disponibles : ['adresse', 'agence', 'chauffage', 'climatisation', 'cuisine', 'dateAnnonce', 'description', 'gouvernorat', 'link', 'nbpiece', 'other_data', 'price', 'reference', 'salle_de_bain', 'superficie_terrain', 'tel', 'typeImm_regroupe', 'contient_corrompu', 'superficie_habitale', 'jardin', 'parking', 'piscine', 'terrasse', 'vue_mer', 'ascenseur', 'type_transaction']

   Utilisation des colonnes : date='dateAnnonce', prix='price', gouvernorat='gouvernorat'

   Après filtrage (date + prix + période 2022+) : 14,409 annonces
   Période : 2022-01-03 → 2023-12-02
   Prix médian : 320,000 DT

   Série temporelle nationale : 24 mois
   Gouvernorats retenus avec données : Ariana, Ben Arous, Bizerte, Manouba, Monastir, Nabeul, Sfax, Sousse, Tunis, Zaghouan

   Aperçu série nationale (p

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 1.  EXPLORATION ET VISUALISATION DES DONNÉES RÉELLES
# ══════════════════════════════════════════════════════════════════════════════
print("\n📊 Figure 1 – Exploration des données réelles...")

fig = plt.figure(figsize=(18, 16), constrained_layout=True)
fig.suptitle("📊 Exploration du Dataset Réel – Immobilier Tunisie (2022–2023)",
             fontsize=16, fontweight="bold", color=PALETTE["dark"])
gs = gridspec.GridSpec(3, 2, figure=fig)

# Volume annonces par mois
ax1 = fig.add_subplot(gs[0, :])
vol_monthly = (df.set_index('dateAnnonce')
                 .resample('MS')['price']
                 .count()
                 .reindex(national_monthly.index, fill_value=0))
ax1.bar(national_monthly.index, vol_monthly.values,
        color=PALETTE["secondary"], alpha=0.7, width=20, label="Nb annonces/mois")
ax1b = ax1.twinx()
ax1b.plot(national_monthly.index, series_main.values,
          color=PALETTE["accent"], lw=2.5, marker='o', markersize=5,
          label="Prix médian (DT)")
ax1.set_title("Volume d'Annonces & Prix Médian Mensuel National", fontweight="bold")
ax1.set_ylabel("Nombre d'annonces", color=PALETTE["secondary"])
ax1b.set_ylabel("Prix médian (DT)", color=PALETTE["accent"])
lines1, lab1 = ax1.get_legend_handles_labels()
lines2, lab2 = ax1b.get_legend_handles_labels()
ax1.legend(lines1+lines2, lab1+lab2, loc="upper left")
ax1.grid(True, alpha=0.3)

# Distribution des prix
ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(df['price'], bins=80, color=PALETTE["primary"], edgecolor="white", alpha=0.8)
ax2.axvline(df['price'].median(), color=PALETTE["accent"], lw=2,
            label=f"Médiane: {df['price'].median():,.0f} DT")
ax2.axvline(df['price'].mean(), color=PALETTE["red"], lw=2, ls='--',
            label=f"Moyenne: {df['price'].mean():,.0f} DT")
ax2.set_title("Distribution des Prix (toutes annonces)", fontweight="bold")
ax2.set_xlabel("Prix (DT)")
ax2.set_ylabel("Fréquence")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Type de transaction
ax3 = fig.add_subplot(gs[1, 1])
trans = df['type_transaction'].value_counts()
colors_t = [PALETTE["primary"], PALETTE["secondary"], PALETTE["accent"]]
wedges, texts, autotexts = ax3.pie(trans.values, labels=trans.index,
                                    autopct='%1.1f%%', colors=colors_t,
                                    startangle=90, textprops={'fontsize': 10})
ax3.set_title("Répartition par Type de Transaction", fontweight="bold")

# Prix par gouvernorat (boîtes)
ax4 = fig.add_subplot(gs[2, :])
gov_price_data = []
gov_labels = []
for gov in GOUVERNORATS_RETENUS:
    sub = df[df['gouvernorat'].str.strip().str.title() == gov.title()]
    if len(sub) == 0:
        sub = df[df['gouvernorat'] == gov]
    if len(sub) > 10:
        gov_price_data.append(sub['price'].clip(0, 1_500_000).values)
        gov_labels.append(f"{gov}\n(n={len(sub):,})")
bp = ax4.boxplot(gov_price_data, labels=gov_labels, patch_artist=True,
                 boxprops=dict(facecolor=PALETTE["secondary"]+"55"),
                 medianprops=dict(color=PALETTE["accent"], lw=2.5))
ax4.set_title("Distribution des Prix par Gouvernorat", fontweight="bold")
ax4.set_ylabel("Prix (DT)")
ax4.tick_params(axis='x', labelsize=8)
ax4.grid(True, alpha=0.3, axis='y')

plt.savefig(f"{OUTPUT_DIR}/01_exploration_reelle.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 01_exploration_reelle.png")


📊 Figure 1 – Exploration des données réelles...
   ✅ 01_exploration_reelle.png


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 2.  DÉCOMPOSITION DE LA SÉRIE TEMPORELLE
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Figure 2 – Décomposition saisonnière...")

def manual_decompose(series, period=12):
    y = series.values.copy().astype(float)
    N = len(y)
    half = period // 2
    trend = np.full(N, np.nan)
    for i in range(half, N - half):
        trend[i] = np.mean(y[i-half:i+half+1])
    # Extrapoler les bords
    if not np.all(np.isnan(trend)):
        valid = np.where(~np.isnan(trend))[0]
        trend[:valid[0]]  = trend[valid[0]]
        trend[valid[-1]:] = trend[valid[-1]]
    detrended = y - trend
    seasonal = np.zeros(N)
    for m in range(period):
        idx = np.arange(m, N, period)
        vals = detrended[idx]
        seasonal[idx] = np.nanmean(vals[~np.isnan(vals)]) if len(vals) > 0 else 0
    residual = y - trend - seasonal
    return trend, seasonal, residual

trend_vals, seasonal_vals, residual_vals = manual_decompose(series_main, min(6, n//2))

fig, axes = plt.subplots(4, 1, figsize=(16, 14), constrained_layout=True)
fig.suptitle("🔍 Décomposition de la Série – Prix Médian National (Données Réelles)",
             fontsize=15, fontweight="bold")
components = [
    (series_main.values,  "Série Originale (Prix Médian DT)",  PALETTE["primary"]),
    (trend_vals,          "Tendance",                          PALETTE["accent"]),
    (seasonal_vals,       "Saisonnalité",                      PALETTE["secondary"]),
    (residual_vals,       "Résidus",                           PALETTE["red"]),
]
for ax, (vals, title, color) in zip(axes, components):
    ax.plot(series_main.index, vals, color=color, lw=2)
    ax.fill_between(series_main.index, vals, alpha=0.1, color=color)
    ax.set_title(title, fontweight="bold")
    ax.grid(True, alpha=0.3)
    if "Prix" in title or "Tendance" in title:
        ax.set_ylabel("DT")

# Annotation de la tendance
z = np.polyfit(np.arange(n), series_main.values, 1)
trend_note = f"Tendance : {'+' if z[0]>0 else ''}{z[0]:.0f} DT/mois ({'+' if z[0]*12>0 else ''}{z[0]*12:.0f} DT/an)"
axes[1].text(0.02, 0.9, trend_note, transform=axes[1].transAxes, fontsize=10,
             bbox=dict(boxstyle="round", fc="white", alpha=0.8))

plt.savefig(f"{OUTPUT_DIR}/02_decomposition_reelle.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 02_decomposition_reelle.png")

📊 Figure 2 – Décomposition saisonnière...
   ✅ 02_decomposition_reelle.png


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# HELPERS : TRAIN/TEST SPLIT & MÉTRIQUES
# (24 mois → 18 train / 6 test — adapté à la taille réelle)
# ══════════════════════════════════════════════════════════════════════════════
TRAIN_SIZE = 18
TEST_SIZE  = n - TRAIN_SIZE   # 6 mois

train_series = series_main.iloc[:TRAIN_SIZE]
test_series  = series_main.iloc[TRAIN_SIZE:]
train_dates  = series_main.index[:TRAIN_SIZE]
test_dates   = series_main.index[TRAIN_SIZE:]
HORIZON      = TEST_SIZE

print(f"\n⚙️ Split : {TRAIN_SIZE} mois train / {TEST_SIZE} mois test")
print(f"   Train : {train_dates[0].date()} → {train_dates[-1].date()}")
print(f"   Test  : {test_dates[0].date()}  → {test_dates[-1].date()}")

def eval_metrics(actual, predicted, model_name):
    actual    = np.array(actual, dtype=float)
    predicted = np.array(predicted, dtype=float)
    mae  = mean_absolute_error(actual, predicted)
    rmse = math.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / (actual + 1e-8))) * 100
    return {"Modèle": model_name, "MAE (DT)": mae, "RMSE (DT)": rmse, "MAPE (%)": mape}

results_all = []


⚙️ Split : 18 mois train / 6 mois test
   Train : 2022-01-01 → 2023-06-01
   Test  : 2023-07-01  → 2023-12-01


## 📈 3. Modèle ARIMA(2,1,1)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 3.  MODÈLE ARIMA(2,1,1)
# ══════════════════════════════════════════════════════════════════════════════
print("\n🔧 Modèle ARIMA(2,1,1)...")

class SimpleARIMA:
    def __init__(self, p=2, d=1, q=1):
        self.p, self.d, self.q = p, d, q

    def difference(self, y, d):
        for _ in range(d): y = np.diff(y)
        return y

    def fit(self, y):
        self.orig = np.array(y, dtype=float)
        diff_y = self.difference(self.orig, self.d)
        self.mean_diff = diff_y.mean()
        diff_y = diff_y - self.mean_diff
        N = len(diff_y)
        self.residuals = np.zeros(N)
        if self.p > 0 and N > self.p:
            X = np.column_stack([diff_y[i:N-self.p+i] for i in range(self.p)])
            Y = diff_y[self.p:]
            self.ar_coefs = np.linalg.lstsq(X, Y, rcond=None)[0]
            pred_ar = X @ self.ar_coefs
            self.residuals[self.p:] = Y - pred_ar
        else:
            self.ar_coefs = np.zeros(self.p)
        return self

    def forecast(self, steps):
        diff_y = self.difference(self.orig, self.d) - self.mean_diff
        history  = list(diff_y)
        res_hist = list(self.residuals)
        forecasts_diff = []
        for _ in range(steps):
            ar_term = sum(self.ar_coefs[i] * history[-(i+1)]
                         for i in range(self.p)) if self.p > 0 else 0
            ma_term = res_hist[-1] * 0.3 if self.q > 0 else 0
            pred = ar_term + ma_term + self.mean_diff
            forecasts_diff.append(pred)
            history.append(pred); res_hist.append(0)
        result = []
        last = self.orig[-1]
        for v in forecasts_diff:
            last = last + v
            result.append(last)
        return np.array(result)

arima = SimpleARIMA(p=2, d=1, q=1)
arima.fit(train_series.values)
arima_pred = arima.forecast(HORIZON)
arima_pred = np.clip(arima_pred, series_main.min()*0.7, series_main.max()*1.5)

results_all.append({
    "model": "ARIMA(2,1,1)",
    "pred":  arima_pred,
    "color": PALETTE["blue"],
    "metrics": eval_metrics(test_series.values, arima_pred, "ARIMA(2,1,1)")
})
m = results_all[-1]['metrics']
print(f"   MAE={m['MAE (DT)']:,.0f} DT | MAPE={m['MAPE (%)']:.2f}%")


🔧 Modèle ARIMA(2,1,1)...
   MAE=50,971 DT | MAPE=15.28%


## 📈 4. Modèle SARIMA(1,1,1)(s=6)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 4.  MODÈLE SARIMA(1,1,1)(1,0,1,6)  –  période 6 car seulement 18 pts train
# ══════════════════════════════════════════════════════════════════════════════
print("🔧 Modèle SARIMA(1,1,1)(1,0,1,6)...")

class SimpleSARIMA:
    def __init__(self, p=1, d=1, q=1, s=6):
        self.p, self.d, self.q, self.s = p, d, q, s
        self.arima = SimpleARIMA(p, d, q)

    def fit(self, y):
        y = np.array(y, dtype=float)
        s = self.s
        self.seasonal_pattern = np.array([np.mean(y[m::s]) for m in range(s)])
        self.last_seasonal = y[-s:]
        diff_s = y[s:] - y[:-s]
        if len(diff_s) > self.p + 1:
            self.arima.fit(diff_s)
        else:
            self.arima.fit(y)
        return self

    def forecast(self, steps):
        try:
            arima_f = self.arima.forecast(steps)
        except:
            arima_f = np.zeros(steps)
        result = []
        for i, v in enumerate(arima_f):
            result.append(v + self.last_seasonal[i % self.s])
        return np.array(result)

sarima = SimpleSARIMA(p=1, d=1, q=1, s=6)
sarima.fit(train_series.values)
sarima_pred = sarima.forecast(HORIZON)
sarima_pred = np.clip(sarima_pred, series_main.min()*0.7, series_main.max()*1.5)

results_all.append({
    "model": "SARIMA(1,1,1)(s=6)",
    "pred":  sarima_pred,
    "color": PALETTE["accent"],
    "metrics": eval_metrics(test_series.values, sarima_pred, "SARIMA(1,1,1)(s=6)")
})
m = results_all[-1]['metrics']
print(f"   MAE={m['MAE (DT)']:,.0f} DT | MAPE={m['MAPE (%)']:.2f}%")

🔧 Modèle SARIMA(1,1,1)(1,0,1,6)...
   MAE=84,028 DT | MAPE=23.65%


## 📈 5. Modèle Prophet


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 5.  PROPHET (régression additive Fourier + trend linéaire)
# ══════════════════════════════════════════════════════════════════════════════
print("🔧 Modèle Prophet (décomposition additive)...")

class SimpleProphet:
    def __init__(self, fourier_order=3):
        self.k = fourier_order

    def _fourier_features(self, t, T, k):
        feats = []
        for i in range(1, k+1):
            feats.append(np.sin(2 * np.pi * i * t / T))
            feats.append(np.cos(2 * np.pi * i * t / T))
        return np.column_stack(feats) if feats else np.ones((len(t), 1))

    def fit(self, t, y):
        self.T = len(t)
        F = self._fourier_features(t, self.T, self.k)
        X = np.column_stack([np.ones(len(t)), t / max(self.T, 1), F])
        self.coefs = np.linalg.lstsq(X, y, rcond=None)[0]
        return self

    def predict(self, t_future):
        F = self._fourier_features(t_future, self.T, self.k)
        X = np.column_stack([np.ones(len(t_future)), t_future / max(self.T, 1), F])
        return X @ self.coefs

t_train  = np.arange(TRAIN_SIZE, dtype=float)
t_test   = np.arange(TRAIN_SIZE, TRAIN_SIZE + HORIZON, dtype=float)

prophet = SimpleProphet(fourier_order=3)
prophet.fit(t_train, train_series.values)
prophet_pred = prophet.predict(t_test)
prophet_pred = np.clip(prophet_pred, series_main.min()*0.7, series_main.max()*1.5)

results_all.append({
    "model": "Prophet",
    "pred":  prophet_pred,
    "color": PALETTE["green"],
    "metrics": eval_metrics(test_series.values, prophet_pred, "Prophet")
})
m = results_all[-1]['metrics']
print(f"   MAE={m['MAE (DT)']:,.0f} DT | MAPE={m['MAPE (%)']:.2f}%")

🔧 Modèle Prophet (décomposition additive)...
   MAE=85,328 DT | MAPE=23.60%


## 🧠 6. Modèle RNN Vanilla


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 6.  RNN VANILLA (NumPy, look_back=3 adapté aux données courtes)
# ══════════════════════════════════════════════════════════════════════════════
print("🔧 Modèle RNN Vanilla...")

LOOK_BACK = 3   # adapté : données courtes

class VanillaRNN:
    def __init__(self, input_size=1, hidden_size=16, output_size=1, lr=0.005):
        self.hs = hidden_size; self.lr = lr
        s = 0.1
        self.Wx = np.random.randn(hidden_size, input_size) * s
        self.Wh = np.random.randn(hidden_size, hidden_size) * s
        self.bh = np.zeros((hidden_size, 1))
        self.Wy = np.random.randn(output_size, hidden_size) * s
        self.by = np.zeros((output_size, 1))
        self.h_last = np.zeros((hidden_size, 1))

    def forward(self, xs):
        h = np.zeros((self.hs, 1))
        ys, hs = [], []
        for x in xs:
            x = np.array(x).reshape(-1, 1)
            h = np.tanh(self.Wh @ h + self.Wx @ x + self.bh)
            y = self.Wy @ h + self.by
            ys.append(y.copy()); hs.append(h.copy())
        self.h_last = h.copy()
        return ys, hs

    def train_step(self, xs, ys_target):
        ys_pred, hs = self.forward(xs)
        T = len(xs)
        dWy=np.zeros_like(self.Wy); dby=np.zeros_like(self.by)
        dWx=np.zeros_like(self.Wx); dWh=np.zeros_like(self.Wh)
        dbh=np.zeros_like(self.bh); dh_next=np.zeros((self.hs,1))
        loss = 0
        for t in reversed(range(T)):
            dy = ys_pred[t] - np.array(ys_target[t]).reshape(-1,1)
            loss += float(np.sum(dy**2))
            dWy += dy @ hs[t].T; dby += dy
            dh = self.Wy.T @ dy + dh_next
            dt = (1 - hs[t]**2) * dh
            dbh += dt
            dWx += dt @ np.array(xs[t]).reshape(1,-1)
            if t > 0: dWh += dt @ hs[t-1].T
            dh_next = self.Wh.T @ dt
        clip = 1.0
        for p, g in [(self.Wy,dWy),(self.by,dby),(self.Wx,dWx),
                     (self.Wh,dWh),(self.bh,dbh)]:
            n2 = np.linalg.norm(g)
            if n2 > clip: g *= clip/n2
            p -= self.lr * g
        return loss / T

    def predict_seq(self, x_last, steps):
        h = self.h_last.copy()
        preds = []
        x = np.array([[x_last]])
        for _ in range(steps):
            h = np.tanh(self.Wh @ h + self.Wx @ x + self.bh)
            y = self.Wy @ h + self.by
            preds.append(float(y.flatten()[0]))
            x = y.reshape(1,-1)
        return np.array(preds)

def train_and_forecast_rnn(model, scaler, train_vals, look_back, epochs):
    sc = scaler.transform(train_vals.reshape(-1,1)).flatten()
    losses = []
    for _ in range(epochs):
        total = 0
        model.h_last = np.zeros((model.hs,1))
        for i in range(look_back, len(sc)):
            xs = [np.array([sc[j]]) for j in range(i-look_back, i)]
            ys = [np.array([sc[j]]) for j in range(i-look_back+1, i+1)]
            total += model.train_step(xs, ys)
        losses.append(total)
    # Warm-up
    model.h_last = np.zeros((model.hs,1))
    model.forward([np.array([sc[j]]) for j in range(-look_back, 0)])
    preds_sc = model.predict_seq(float(sc[-1]), HORIZON)
    preds = scaler.inverse_transform(preds_sc.reshape(-1,1)).flatten()
    return preds, losses

scaler_rnn = MinMaxScaler()
scaler_rnn.fit(train_series.values.reshape(-1,1))
rnn_model = VanillaRNN(hidden_size=16, lr=0.005)
rnn_pred, rnn_losses = train_and_forecast_rnn(rnn_model, scaler_rnn,
                                               train_series.values, LOOK_BACK, 300)
rnn_pred = np.clip(rnn_pred, series_main.min()*0.7, series_main.max()*1.5)

results_all.append({
    "model": "RNN Vanilla",
    "pred":  rnn_pred,
    "color": PALETTE["purple"],
    "metrics": eval_metrics(test_series.values, rnn_pred, "RNN Vanilla"),
    "losses": rnn_losses,
})
m = results_all[-1]['metrics']
print(f"   MAE={m['MAE (DT)']:,.0f} DT | MAPE={m['MAPE (%)']:.2f}%")

🔧 Modèle RNN Vanilla...
   MAE=53,672 DT | MAPE=16.62%


## 🧠 7. Modèle LSTM


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 7.  LSTM
# ══════════════════════════════════════════════════════════════════════════════
print("🔧 Modèle LSTM...")

class SimpleLSTM:
    def __init__(self, input_size=1, hidden_size=16, output_size=1, lr=0.003):
        self.hs = hidden_size; self.lr = lr
        s = 0.05
        self.Wf=np.random.randn(hidden_size,input_size+hidden_size)*s; self.bf=np.ones((hidden_size,1))*0.1
        self.Wi=np.random.randn(hidden_size,input_size+hidden_size)*s; self.bi=np.zeros((hidden_size,1))
        self.Wg=np.random.randn(hidden_size,input_size+hidden_size)*s; self.bg=np.zeros((hidden_size,1))
        self.Wo=np.random.randn(hidden_size,input_size+hidden_size)*s; self.bo=np.zeros((hidden_size,1))
        self.Wy=np.random.randn(output_size,hidden_size)*s;            self.by=np.zeros((output_size,1))
        self.h_last=np.zeros((hidden_size,1)); self.c_last=np.zeros((hidden_size,1))

    @staticmethod
    def sig(x): return 1/(1+np.exp(-np.clip(x,-15,15)))

    def step(self, x, h, c):
        x = np.array(x).reshape(-1,1)
        xh = np.vstack([x, h])
        f=self.sig(self.Wf@xh+self.bf); i=self.sig(self.Wi@xh+self.bi)
        g=np.tanh(self.Wg@xh+self.bg); o=self.sig(self.Wo@xh+self.bo)
        c_new=f*c+i*g; h_new=o*np.tanh(c_new)
        y=self.Wy@h_new+self.by
        return y, h_new, c_new, (f,i,g,o,xh,c,h)

    def train_step(self, xs, ys_t):
        T=len(xs); h=np.zeros((self.hs,1)); c=np.zeros((self.hs,1))
        cache=[]; preds=[]
        for t in range(T):
            y,h,c,cc=self.step(xs[t],h,c)
            cache.append((h.copy(),c.copy(),cc)); preds.append(y.copy())
        self.h_last,self.c_last=h.copy(),c.copy()
        dWf=np.zeros_like(self.Wf);dbf=np.zeros_like(self.bf)
        dWi=np.zeros_like(self.Wi);dbi=np.zeros_like(self.bi)
        dWg=np.zeros_like(self.Wg);dbg=np.zeros_like(self.bg)
        dWo=np.zeros_like(self.Wo);dbo=np.zeros_like(self.bo)
        dWy=np.zeros_like(self.Wy);dby=np.zeros_like(self.by)
        dh_n=np.zeros((self.hs,1)); dc_n=np.zeros((self.hs,1)); loss=0
        for t in reversed(range(T)):
            dy=preds[t]-np.array(ys_t[t]).reshape(-1,1)
            loss+=float(np.sum(dy**2))
            h_t,c_t,(f,i,g,o,xh,c_prev,h_prev)=cache[t]
            dWy+=dy@h_t.T; dby+=dy
            dh=self.Wy.T@dy+dh_n
            dc=dh*o*(1-np.tanh(c_t)**2)+dc_n
            df=dc*c_prev; di=dc*g; dg=dc*i; do_=dh*np.tanh(c_t)
            dWf+=(df*f*(1-f))@xh.T; dbf+=df*f*(1-f)
            dWi+=(di*i*(1-i))@xh.T; dbi+=di*i*(1-i)
            dWg+=(dg*(1-g**2))@xh.T; dbg+=dg*(1-g**2)
            dWo+=(do_*o*(1-o))@xh.T; dbo+=do_*o*(1-o)
            inp_size=len(xs[0])
            dh_n=(self.Wf.T@(df*f*(1-f))+self.Wi.T@(di*i*(1-i))+
                  self.Wg.T@(dg*(1-g**2))+self.Wo.T@(do_*o*(1-o)))[inp_size:]
            dc_n=dc*f
        clip=1.0
        for p,g2 in [(self.Wf,dWf),(self.bf,dbf),(self.Wi,dWi),(self.bi,dbi),
                     (self.Wg,dWg),(self.bg,dbg),(self.Wo,dWo),(self.bo,dbo),
                     (self.Wy,dWy),(self.by,dby)]:
            n2=np.linalg.norm(g2)
            if n2>clip: g2*=clip/n2
            p-=self.lr*g2
        return loss/T

    def predict_seq(self, x_last, steps):
        h,c=self.h_last.copy(),self.c_last.copy()
        preds=[]; x=np.array([[x_last]])
        for _ in range(steps):
            y,h,c,_=self.step(x.flatten(),h,c)
            preds.append(float(y.flatten()[0])); x=y.reshape(1,-1)
        return np.array(preds)

def train_and_forecast_lstm(model, scaler, train_vals, look_back, epochs):
    sc=scaler.transform(train_vals.reshape(-1,1)).flatten()
    losses=[]
    for _ in range(epochs):
        total=0
        model.h_last=np.zeros((model.hs,1)); model.c_last=np.zeros((model.hs,1))
        for i in range(look_back, len(sc)):
            xs=[np.array([sc[j]]) for j in range(i-look_back,i)]
            ys=[np.array([sc[j]]) for j in range(i-look_back+1,i+1)]
            total+=model.train_step(xs,ys)
        losses.append(total)
    model.h_last=np.zeros((model.hs,1)); model.c_last=np.zeros((model.hs,1))
    for j in range(-look_back,0):
        _,model.h_last,model.c_last,_=model.step(np.array([sc[j]]),model.h_last,model.c_last)
    preds_sc=model.predict_seq(float(sc[-1]),HORIZON)
    return scaler.inverse_transform(preds_sc.reshape(-1,1)).flatten(), losses

scaler_lstm=MinMaxScaler()
scaler_lstm.fit(train_series.values.reshape(-1,1))
lstm_model=SimpleLSTM(hidden_size=16, lr=0.003)
lstm_pred, lstm_losses = train_and_forecast_lstm(lstm_model, scaler_lstm,
                                                  train_series.values, LOOK_BACK, 300)
lstm_pred = np.clip(lstm_pred, series_main.min()*0.7, series_main.max()*1.5)

results_all.append({
    "model": "LSTM",
    "pred":  lstm_pred,
    "color": PALETTE["orange"],
    "metrics": eval_metrics(test_series.values, lstm_pred, "LSTM"),
    "losses": lstm_losses,
})
m = results_all[-1]['metrics']
print(f"   MAE={m['MAE (DT)']:,.0f} DT | MAPE={m['MAPE (%)']:.2f}%")

🔧 Modèle LSTM...
   MAE=52,978 DT | MAPE=16.65%


## 🧠 8. Modèle GRU


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 8.  GRU
# ══════════════════════════════════════════════════════════════════════════════
print("🔧 Modèle GRU...")

class SimpleGRU:
    def __init__(self, input_size=1, hidden_size=16, output_size=1, lr=0.003):
        self.hs=hidden_size; self.lr=lr; s=0.05
        self.Wz=np.random.randn(hidden_size,input_size+hidden_size)*s; self.bz=np.zeros((hidden_size,1))
        self.Wr=np.random.randn(hidden_size,input_size+hidden_size)*s; self.br=np.zeros((hidden_size,1))
        self.Wh=np.random.randn(hidden_size,input_size+hidden_size)*s; self.bh=np.zeros((hidden_size,1))
        self.Wy=np.random.randn(output_size,hidden_size)*s;            self.by=np.zeros((output_size,1))
        self.h_last=np.zeros((hidden_size,1))

    @staticmethod
    def sig(x): return 1/(1+np.exp(-np.clip(x,-15,15)))

    def step(self, x, h):
        x=np.array(x).reshape(-1,1); xh=np.vstack([x,h])
        z=self.sig(self.Wz@xh+self.bz); r=self.sig(self.Wr@xh+self.br)
        xrh=np.vstack([x, r*h])
        h_t=np.tanh(self.Wh@xrh+self.bh)
        h_new=(1-z)*h+z*h_t
        y=self.Wy@h_new+self.by
        return y, h_new, (z,r,h_t,xh,xrh,h)

    def train_step(self, xs, ys_t):
        T=len(xs); h=np.zeros((self.hs,1))
        cache=[]; preds=[]
        for t in range(T):
            y,h,cc=self.step(xs[t],h)
            cache.append((h.copy(),cc)); preds.append(y.copy())
        self.h_last=h.copy()
        dWz=np.zeros_like(self.Wz);dbz=np.zeros_like(self.bz)
        dWr=np.zeros_like(self.Wr);dbr=np.zeros_like(self.br)
        dWh_=np.zeros_like(self.Wh);dbh_=np.zeros_like(self.bh)
        dWy=np.zeros_like(self.Wy);dby=np.zeros_like(self.by)
        dh_n=np.zeros((self.hs,1)); loss=0
        for t in reversed(range(T)):
            dy=preds[t]-np.array(ys_t[t]).reshape(-1,1)
            loss+=float(np.sum(dy**2))
            h_t_,(z,r,h_tilde,xh,xrh,h_prev)=cache[t]
            dWy+=dy@h_t_.T; dby+=dy
            dh=self.Wy.T@dy+dh_n
            dht=dh*z*(1-h_tilde**2)
            dz=dh*(h_tilde-h_prev)*z*(1-z)
            dWz+=dz@xh.T; dbz+=dz
            dWh_+=dht@xrh.T; dbh_+=dht
            inp_size=len(xs[0])
            dr=(self.Wh.T@dht)[inp_size:]*h_prev*r*(1-r)
            dWr+=dr@xh.T; dbr+=dr
            dh_n=dh*(1-z)+(self.Wz.T@dz)[inp_size:]
        clip=1.0
        for p,g2 in [(self.Wz,dWz),(self.bz,dbz),(self.Wr,dWr),(self.br,dbr),
                     (self.Wh,dWh_),(self.bh,dbh_),(self.Wy,dWy),(self.by,dby)]:
            n2=np.linalg.norm(g2)
            if n2>clip: g2*=clip/n2
            p-=self.lr*g2
        return loss/T

    def predict_seq(self, x_last, steps):
        h=self.h_last.copy(); preds=[]; x=np.array([[x_last]])
        for _ in range(steps):
            y,h,_=self.step(x.flatten(),h)
            preds.append(float(y.flatten()[0])); x=y.reshape(1,-1)
        return np.array(preds)

def train_and_forecast_gru(model, scaler, train_vals, look_back, epochs):
    sc=scaler.transform(train_vals.reshape(-1,1)).flatten()
    losses=[]
    for _ in range(epochs):
        total=0; model.h_last=np.zeros((model.hs,1))
        for i in range(look_back, len(sc)):
            xs=[np.array([sc[j]]) for j in range(i-look_back,i)]
            ys=[np.array([sc[j]]) for j in range(i-look_back+1,i+1)]
            total+=model.train_step(xs,ys)
        losses.append(total)
    model.h_last=np.zeros((model.hs,1))
    for j in range(-look_back,0):
        _,model.h_last,_=model.step(np.array([sc[j]]),model.h_last)
    preds_sc=model.predict_seq(float(sc[-1]),HORIZON)
    return scaler.inverse_transform(preds_sc.reshape(-1,1)).flatten(), losses

scaler_gru=MinMaxScaler()
scaler_gru.fit(train_series.values.reshape(-1,1))
gru_model=SimpleGRU(hidden_size=16, lr=0.003)
gru_pred, gru_losses = train_and_forecast_gru(gru_model, scaler_gru,
                                               train_series.values, LOOK_BACK, 300)
gru_pred = np.clip(gru_pred, series_main.min()*0.7, series_main.max()*1.5)

results_all.append({
    "model": "GRU",
    "pred":  gru_pred,
    "color": PALETTE["secondary"],
    "metrics": eval_metrics(test_series.values, gru_pred, "GRU"),
    "losses": gru_losses,
})
m = results_all[-1]['metrics']
print(f"   MAE={m['MAE (DT)']:,.0f} DT | MAPE={m['MAPE (%)']:.2f}%")

🔧 Modèle GRU...
   MAE=53,015 DT | MAPE=16.66%


## 📊 9. Prédictions – Visualisation Comparative


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 9.  VISUALISATION – PRÉDICTIONS DE CHAQUE MODÈLE
# ══════════════════════════════════════════════════════════════════════════════
print("\n📊 Figure 3 – Prédictions par modèle...")

fig, axes = plt.subplots(3, 2, figsize=(18, 14), constrained_layout=True)
fig.suptitle("📈 Prédictions des 6 Modèles – Prix Médian National (Données Réelles)",
             fontsize=16, fontweight="bold")

for ax, res in zip(axes.flatten(), results_all):
    ax.plot(train_dates, train_series, color=PALETTE["dark"], lw=2,
            marker='o', markersize=4, alpha=0.8, label="Train (réel)")
    ax.plot(test_dates,  test_series,  color=PALETTE["dark"], lw=2.5,
            ls="--", marker='s', markersize=5, label="Test (réel)")
    ax.plot(test_dates,  res["pred"],  color=res["color"], lw=2.5,
            marker='^', markersize=5, label=f"Prédit")
    ax.fill_between(test_dates,
                    res["pred"] * 0.95, res["pred"] * 1.05,
                    alpha=0.2, color=res["color"])
    m = res["metrics"]
    info = (f"MAE = {m['MAE (DT)']:,.0f} DT\n"
            f"RMSE = {m['RMSE (DT)']:,.0f} DT\n"
            f"MAPE = {m['MAPE (%)']:.2f}%")
    ax.text(0.02, 0.97, info, transform=ax.transAxes, va="top", fontsize=9,
            bbox=dict(boxstyle="round,pad=0.4", fc="white", alpha=0.9))
    ax.axvline(test_dates[0], color="gray", ls=":", lw=1.5)
    ax.set_title(res["model"], fontweight="bold", color=res["color"])
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)
    ax.set_ylabel("Prix médian (DT)")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))

plt.savefig(f"{OUTPUT_DIR}/03_predictions_reelles.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 03_predictions_reelles.png")


📊 Figure 3 – Prédictions par modèle...
   ✅ 03_predictions_reelles.png


## 🏆 10. Comparaison des Métriques


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 10.  COMPARAISON DES MÉTRIQUES
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Figure 4 – Comparaison métriques...")

metrics_df = pd.DataFrame([r["metrics"] for r in results_all]).sort_values("MAPE (%)")
model_color_map = {r["model"]: r["color"] for r in results_all}
colors_sorted = [model_color_map[m] for m in metrics_df["Modèle"]]

fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)
fig.suptitle("🏆 Comparaison des Performances – Données Réelles",
             fontsize=15, fontweight="bold")

for ax, metric in zip(axes, ["MAE (DT)", "RMSE (DT)", "MAPE (%)"]):
    bars = ax.bar(metrics_df["Modèle"], metrics_df[metric],
                  color=colors_sorted, edgecolor="white", lw=1.5, width=0.6)
    ax.set_title(metric, fontweight="bold")
    ax.set_ylabel(metric)
    ax.set_xticklabels(metrics_df["Modèle"], rotation=30, ha="right", fontsize=9)
    ax.grid(True, alpha=0.3, axis="y")
    # Valeur sur barre
    for bar, val in zip(bars, metrics_df[metric]):
        label = f"{val:,.0f}" if "DT" in metric else f"{val:.2f}%"
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+max(metrics_df[metric])*0.01,
                label, ha="center", va="bottom", fontsize=8, fontweight="bold")
    # Encadrer le meilleur
    best_model = metrics_df.iloc[0]["Modèle"]
    idx = list(metrics_df["Modèle"]).index(best_model)
    bars[idx].set_edgecolor(PALETTE["dark"]); bars[idx].set_linewidth(3)

plt.savefig(f"{OUTPUT_DIR}/04_comparaison_metriques_reelles.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 04_comparaison_metriques_reelles.png")

📊 Figure 4 – Comparaison métriques...
   ✅ 04_comparaison_metriques_reelles.png


## 📉 11. Courbes d'Entraînement (Deep Learning)


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 11.  COURBES DE PERTE (DL)
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Figure 5 – Courbes d'entraînement...")

fig, axes = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
fig.suptitle("📉 Courbes d'Entraînement – Modèles Deep Learning (Données Réelles)",
             fontsize=14, fontweight="bold")
dl_models = [r for r in results_all if "losses" in r]
for ax, res in zip(axes, dl_models):
    ls = np.array(res["losses"])
    ax.plot(ls, color=res["color"], alpha=0.35, lw=1)
    k = min(15, len(ls)//3)
    if k > 0:
        smooth = np.convolve(ls, np.ones(k)/k, mode='valid')
        ax.plot(range(k-1, len(ls)), smooth, color=res["color"], lw=2.5, label="Lissée")
    ax.set_title(res["model"], fontweight="bold", color=res["color"])
    ax.set_xlabel("Époque"); ax.set_ylabel("MSE Loss")
    ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

plt.savefig(f"{OUTPUT_DIR}/05_training_loss_reelles.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 05_training_loss_reelles.png")

📊 Figure 5 – Courbes d'entraînement...
   ✅ 05_training_loss_reelles.png


## 📅 12. Timing Achat/Vente par Gouvernorat


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 12.  TIMING ACHAT / VENTE PAR GOUVERNORAT (données réelles)
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Figure 6 – Timing achat/vente par gouvernorat...")

fig, axes = plt.subplots(2, 5, figsize=(24, 10), constrained_layout=True)
fig.suptitle("📅 Saisonnalité Mensuelle des Prix – Données Réelles\nMeilleur Moment Achat/Vente par Gouvernorat",
             fontsize=15, fontweight="bold")

rendements = {}
for ax, gov in zip(axes.flatten(), GOUVERNORATS_RETENUS):
    s = gov_monthly[gov]
    # Extraire mois
    monthly_avg = np.array([s[s.index.month == m+1].mean() for m in range(12)])
    monthly_avg = np.where(np.isnan(monthly_avg), np.nanmean(monthly_avg), monthly_avg)

    min_m = int(np.argmin(monthly_avg))
    max_m = int(np.argmax(monthly_avg))
    rendements[gov] = (monthly_avg.max() - monthly_avg.min()) / (monthly_avg.min() + 1e-8) * 100

    colors_m = [PALETTE["green"] if i == min_m else
                PALETTE["red"]   if i == max_m else
                PALETTE["secondary"] for i in range(12)]
    ax.bar(MONTH_NAMES, monthly_avg/1000, color=colors_m, edgecolor="white", lw=1)
    ax.axhline(np.nanmean(monthly_avg)/1000, color=PALETTE["dark"], ls="--", lw=1.2)
    ax.set_title(gov, fontweight="bold")
    ax.set_ylabel("Prix (kDT)")
    ax.set_xticklabels(MONTH_NAMES, rotation=45, fontsize=7)
    ax.grid(True, alpha=0.3, axis="y")
    patch_b = mpatches.Patch(color=PALETTE["green"], label=f"Acheter: {MONTH_NAMES[min_m]}")
    patch_s = mpatches.Patch(color=PALETTE["red"],   label=f"Vendre: {MONTH_NAMES[max_m]}")
    ax.legend(handles=[patch_b, patch_s], fontsize=7)

plt.savefig(f"{OUTPUT_DIR}/06_timing_achat_vente_reel.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 06_timing_achat_vente_reel.png")

📊 Figure 6 – Timing achat/vente par gouvernorat...
   ✅ 06_timing_achat_vente_reel.png


## 🔮 13. Prévisions Futures + Heatmap Localisation


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 13.  PRÉVISIONS FUTURES + HEATMAP LOCALISATION
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Figure 7 – Prévisions futures + heatmap...")

best_res  = min(results_all, key=lambda r: r["metrics"]["MAPE (%)"])
FUTURE_6M = 6
future_dates = pd.date_range(series_main.index[-1] + pd.offsets.MonthBegin(1),
                              periods=FUTURE_6M, freq="MS")

arima_full = SimpleARIMA(p=2, d=1, q=1)
arima_full.fit(series_main.values)
future_pred = arima_full.forecast(FUTURE_6M)
future_pred = np.clip(future_pred, series_main.min()*0.8, series_main.max()*1.4)

fig = plt.figure(figsize=(18, 12), constrained_layout=True)
fig.suptitle("🔮 Prévisions 6 Mois + Analyse Localisation – Données Réelles",
             fontsize=15, fontweight="bold")
gs2 = gridspec.GridSpec(2, 2, figure=fig)

ax1 = fig.add_subplot(gs2[0, :])
ax1.plot(series_main.index, series_main.values/1000,
         color=PALETTE["dark"], lw=2.5, marker='o', markersize=5, label="Historique réel")
ax1.plot(test_dates, best_res["pred"]/1000, color=best_res["color"], lw=2, ls="--",
         label=f"Test prédit – {best_res['model']}")
ax1.plot(future_dates, future_pred/1000, color=PALETTE["red"], lw=2.5,
         marker='D', markersize=6, label="Prévision 6 mois (ARIMA)")
ax1.fill_between(future_dates, future_pred*0.93/1000, future_pred*1.07/1000,
                 alpha=0.2, color=PALETTE["red"], label="IC ±7%")
ax1.axvline(test_dates[0], color="gray", ls=":", lw=1.5, label="Début test")
ax1.axvline(future_dates[0], color=PALETTE["red"], ls=":", lw=1.5, label="Début prévision")
ax1.set_title(f"Prix Médian National (kDT) – Prévision jusqu'à {future_dates[-1].strftime('%b %Y')}",
              fontweight="bold")
ax1.legend(fontsize=9, ncol=3); ax1.grid(True, alpha=0.3)
ax1.set_ylabel("Prix médian (kDT)")
# Annoter les valeurs futures
for d, v in zip(future_dates, future_pred):
    ax1.annotate(f"{v/1000:.0f}k", (d, v/1000), textcoords="offset points",
                 xytext=(0, 8), ha='center', fontsize=8, color=PALETTE["red"])

# Heatmap prix par gouvernorat par mois
ax2 = fig.add_subplot(gs2[1, 0])
heatmap_data = np.array([
    [gov_monthly[gov][gov_monthly[gov].index.month == m+1].mean() / 1000
     for m in range(12)]
    for gov in GOUVERNORATS_RETENUS
])
heatmap_data = np.where(np.isnan(heatmap_data), np.nanmean(heatmap_data), heatmap_data)
cmap = LinearSegmentedColormap.from_list("c", [PALETTE["secondary"], PALETTE["primary"], PALETTE["dark"]])
im = ax2.imshow(heatmap_data, cmap=cmap, aspect='auto')
ax2.set_xticks(range(12)); ax2.set_xticklabels(MONTH_NAMES, fontsize=8)
ax2.set_yticks(range(len(GOUVERNORATS_RETENUS)))
ax2.set_yticklabels(GOUVERNORATS_RETENUS, fontsize=9)
ax2.set_title("Heatmap Prix Médian Mensuel (kDT)", fontweight="bold")
plt.colorbar(im, ax=ax2, shrink=0.8, label="kDT")
for i in range(len(GOUVERNORATS_RETENUS)):
    for j in range(12):
        ax2.text(j, i, f"{heatmap_data[i,j]:.0f}", ha='center', va='center',
                 color="white", fontsize=7, fontweight="bold")

# Rendement saisonnier
ax3 = fig.add_subplot(gs2[1, 1])
gov_s = sorted(rendements, key=rendements.get, reverse=True)
ren_v = [rendements[g] for g in gov_s]
mean_ren = np.mean(ren_v)
bar_c = [PALETTE["green"] if v > mean_ren else PALETTE["secondary"] for v in ren_v]
bars = ax3.barh(gov_s, ren_v, color=bar_c, edgecolor="white")
ax3.axvline(mean_ren, color=PALETTE["accent"], ls="--", lw=2,
            label=f"Moyenne: {mean_ren:.1f}%")
ax3.set_title("Rendement Saisonnier Potentiel\n(achat bas → vente haut)", fontweight="bold")
ax3.set_xlabel("Rendement (%)")
ax3.legend(); ax3.grid(True, alpha=0.3, axis="x")
for bar, val in zip(bars, ren_v):
    ax3.text(val+0.2, bar.get_y()+bar.get_height()/2,
             f"+{val:.1f}%", va='center', fontsize=9)

plt.savefig(f"{OUTPUT_DIR}/07_previsions_localisation_reelles.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 07_previsions_localisation_reelles.png")

📊 Figure 7 – Prévisions futures + heatmap...
   ✅ 07_previsions_localisation_reelles.png


## 🏆 14. Tableau de Bord Final


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# 14.  TABLEAU DE BORD FINAL (RADAR + CLASSEMENT + RECOMMANDATIONS)
# ══════════════════════════════════════════════════════════════════════════════
print("📊 Figure 8 – Tableau de bord final...")

speed_scores = {"ARIMA(2,1,1)":10, "SARIMA(1,1,1)(s=6)":9,
                "Prophet":8, "RNN Vanilla":6, "LSTM":5, "GRU":6}
interp_scores= {"ARIMA(2,1,1)":9, "SARIMA(1,1,1)(s=6)":9,
                "Prophet":8, "RNN Vanilla":4, "LSTM":3, "GRU":4}

fig = plt.figure(figsize=(20, 14), constrained_layout=True)
fig.suptitle("🏆 Tableau de Bord Final – Comparaison Modèles (Données Réelles)",
             fontsize=17, fontweight="bold")
gs3 = gridspec.GridSpec(3, 3, figure=fig)

# Radar
ax_r = fig.add_subplot(gs3[:2, :2], polar=True)
categories = ["MAE\n(↓)", "RMSE\n(↓)", "MAPE\n(↓)", "Vitesse\n(↑)", "Interprét.\n(↑)"]
N = len(categories)
angles = [n/float(N)*2*np.pi for n in range(N)]; angles += angles[:1]
max_mae  = metrics_df["MAE (DT)"].max()
max_rmse = metrics_df["RMSE (DT)"].max()
max_mape = metrics_df["MAPE (%)"].max()

for res in results_all:
    m2 = res["metrics"]; mod = m2["Modèle"]
    scores = [
        (1 - m2["MAE (DT)"] / max_mae) * 10,
        (1 - m2["RMSE (DT)"] / max_rmse) * 10,
        (1 - m2["MAPE (%)"] / max_mape) * 10,
        speed_scores.get(mod, 5),
        interp_scores.get(mod, 5),
    ]; scores += scores[:1]
    ax_r.plot(angles, scores, color=res["color"], lw=2, label=mod)
    ax_r.fill(angles, scores, color=res["color"], alpha=0.07)
ax_r.set_xticks(angles[:-1]); ax_r.set_xticklabels(categories, size=10)
ax_r.set_ylim(0, 10)
ax_r.set_title("Radar – Performance Globale", fontweight="bold", pad=20, fontsize=13)
ax_r.legend(loc="lower left", bbox_to_anchor=(-0.25,-0.2), fontsize=9, ncol=2)

# Tableau
ax_t = fig.add_subplot(gs3[:2, 2]); ax_t.axis("off")
table_data = []
sorted_res = sorted(results_all, key=lambda r: r["metrics"]["MAPE (%)"])
for i, res in enumerate(sorted_res):
    m2 = res["metrics"]
    medals = ["🥇","🥈","🥉","  ","  ","  "]
    table_data.append([
        medals[i] + m2["Modèle"],
        f"{m2['MAE (DT)']:,.0f}",
        f"{m2['RMSE (DT)']:,.0f}",
        f"{m2['MAPE (%)']:.2f}%",
    ])
tbl = ax_t.table(cellText=table_data, colLabels=["Modèle","MAE","RMSE","MAPE"],
                 loc="center", cellLoc="center")
tbl.auto_set_font_size(False); tbl.set_fontsize(10); tbl.scale(1.3, 2.0)
for (row,col), cell in tbl.get_celld().items():
    if row == 0:
        cell.set_facecolor(PALETTE["primary"]); cell.set_text_props(color="white", fontweight="bold")
    elif row % 2 == 0: cell.set_facecolor("#EEF7F0")
    if col == 3 and row == 1:
        cell.set_facecolor(PALETTE["green"]); cell.set_text_props(fontweight="bold")
ax_t.set_title("Classement par MAPE ↓", fontweight="bold", fontsize=12, pad=10)

# Recommandations
ax_txt = fig.add_subplot(gs3[2, :]); ax_txt.axis("off")
best_gov_buy  = min(rendements, key=rendements.get)
best_gov_sell = max(rendements, key=rendements.get)
all_monthly_nat = np.array([
    series_main[series_main.index.month == m+1].mean() for m in range(12)
])
best_buy_m  = MONTH_NAMES[int(np.nanargmin(all_monthly_nat))]
best_sell_m = MONTH_NAMES[int(np.nanargmax(all_monthly_nat))]

summary = (
    f"■  MEILLEUR MODÈLE : {best_res['model']}  (MAPE = {best_res['metrics']['MAPE (%)']:.2f}%)   "
    f"─  Données : 42 713 annonces réelles | Période modélisée : 2022–2023 ({n} mois)\n\n"
    f"■  RECOMMANDATION ACHAT  : Mois de {best_buy_m} → prix médians les plus bas (données réelles)\n"
    f"   Gouvernorat le plus accessible : {best_gov_buy} (prix médian bas, fort volume d'offres)\n\n"
    f"■  RECOMMANDATION VENTE  : Mois de {best_sell_m} → pics de prix observés sur les données historiques\n"
    f"   Gouvernorat à fort rendement saisonnier : {best_gov_sell} (+{rendements[best_gov_sell]:.1f}%)\n\n"
    f"■  LIMITE : Série courte (24 mois) — enrichir avec davantage d'annonces datées pour améliorer les DL."
)
ax_txt.text(0.01, 0.95, summary, transform=ax_txt.transAxes, fontsize=10, va="top",
            bbox=dict(boxstyle="round,pad=0.6", fc="#EEF7F0", ec=PALETTE["primary"], lw=2))
ax_txt.set_title("📌 Synthèse & Recommandations DSO (sur données réelles)",
                 fontweight="bold", fontsize=13, color=PALETTE["primary"])

plt.savefig(f"{OUTPUT_DIR}/08_tableau_bord_reel.png", dpi=150, bbox_inches="tight")
plt.close()
print("   ✅ 08_tableau_bord_reel.png")

📊 Figure 8 – Tableau de bord final...
   ✅ 08_tableau_bord_reel.png


## ✅ Résumé Final & Recommandations


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
# RÉSUMÉ
# ══════════════════════════════════════════════════════════════════════════════
print("\n" + "═"*65)
print("  ✅ PIPELINE DONNÉES RÉELLES TERMINÉ")
print("═"*65)
print(f"\n  Dataset : {DATASET_PATH}")
print(f"  Annonces totales   : {df_raw.shape[0]:,}")
print(f"  Annonces utilisées : {len(df):,}  (avec date + prix valide)")
print(f"  Série temporelle   : {n} mois (Jan 2022 – Déc 2023)")
print(f"  Train / Test       : {TRAIN_SIZE} / {TEST_SIZE} mois\n")
print(f"{'Modèle':<30} {'MAE (DT)':>10} {'RMSE (DT)':>10} {'MAPE':>8}")
print("-"*62)
for res in sorted(results_all, key=lambda r: r["metrics"]["MAPE (%)"]):
    m2 = res["metrics"]
    flag = " 🏆" if res == best_res else ""
    print(f"  {m2['Modèle']:<28} {m2['MAE (DT)']:>9,.0f} {m2['RMSE (DT)']:>9,.0f} {m2['MAPE (%)']:>7.2f}%{flag}")
print(f"\n  📅 Acheter en : {best_buy_m}  |  Vendre en : {best_sell_m}")
print(f"  🗺️  Meilleur rendement : {best_gov_sell}  (+{rendements[best_gov_sell]:.1f}%)")
print("═"*65)


═════════════════════════════════════════════════════════════════
  ✅ PIPELINE DONNÉES RÉELLES TERMINÉ
═════════════════════════════════════════════════════════════════

  Dataset : /mnt/user-data/uploads/dataset_immobilier_clean__1_.csv
  Annonces totales   : 42,713
  Annonces utilisées : 14,409  (avec date + prix valide)
  Série temporelle   : 24 mois (Jan 2022 – Déc 2023)
  Train / Test       : 18 / 6 mois

Modèle                           MAE (DT)  RMSE (DT)     MAPE
--------------------------------------------------------------
  ARIMA(2,1,1)                    50,971    55,096   15.28% 🏆
  RNN Vanilla                     53,672    55,476   16.62%
  LSTM                            52,978    55,186   16.65%
  GRU                             53,015    55,207   16.66%
  Prophet                         85,328   107,073   23.60%
  SARIMA(1,1,1)(s=6)              84,028   104,472   23.65%

  📅 Acheter en : Mai  |  Vendre en : Jul
  🗺️  Meilleur rendement : Tunis  (+534.7%)
════════════

In [ ]:
from google.colab import drive
drive.mount('/content/drive')